# LeetCode 933 — Number of Recent Calls bằng Python

**Link đề:** https://leetcode.com/problems/number-of-recent-calls/
**Độ khó:** Easy · **Chủ đề:** Design (thiết kế class), Queue (hàng đợi), Data Stream

```python
class RecentCounter:

    def __init__(self):
        ...

    def ping(self, t: int) -> int:
        ...
```

## Mục tiêu học
1. Hiểu dạng bài **thiết kế (design)**: viết một class có trạng thái, LeetCode gọi các method theo chuỗi lệnh.
2. Nắm nền tảng: `class`, `__init__`, `self`, thuộc tính đối tượng.
3. Học **hàng đợi (queue)** với `collections.deque` và kỹ thuật **cửa sổ trượt theo thời gian** (sliding window).
4. Đi từ cách lưu hết rồi đếm lại → deque bỏ phần tử cũ → tìm kiếm nhị phân `bisect` → con trỏ "đầu hàng" trên list.
5. Phân tích **khấu hao** (amortized) `O(1)` mỗi lần `ping`, trace, test bằng bộ chạy lệnh giả lập LeetCode.

**Cách học:** đọc markdown → đoán output → chạy → so sánh. Cell TODO không tự chạy.

## 1. Đề bài — đếm số request trong 3000 ms gần nhất

Viết class `RecentCounter`:
- `RecentCounter()` khởi tạo bộ đếm, chưa có request nào.
- `ping(t)` thêm một request tại thời điểm `t` (mili-giây) và trả về **số request trong khoảng `[t - 3000, t]`** (tính cả hai đầu), bao gồm request vừa thêm.

Đảm bảo: mỗi lần gọi `ping` có `t` **tăng nghiêm ngặt**.

Ví dụ (định dạng LeetCode):
```text
Input : ["RecentCounter", "ping", "ping", "ping", "ping"]
        [[], [1], [100], [3001], [3002]]
Output: [null, 1, 2, 3, 3]
```

| Lệnh | requests sau khi thêm | Khoảng `[t-3000, t]` | Trả về |
|---|---|---|:---:|
| `ping(1)` | `[1]` | `[-2999, 1]` | 1 |
| `ping(100)` | `[1, 100]` | `[-2900, 100]` | 2 |
| `ping(3001)` | `[1, 100, 3001]` | `[1, 3001]` | 3 |
| `ping(3002)` | `[1, 100, 3001, 3002]` | `[2, 3002]` | 3 (request `1` đã rơi ra) |

**Constraints:** `1 <= t <= 10^9`; `t` tăng nghiêm ngặt; tối đa `10^4` lần gọi `ping`.

## 2. Nền tảng Python cần dùng

### 2.1. `class`, `__init__`, `self` — đối tượng có trạng thái

- `class` định nghĩa một **kiểu dữ liệu mới**.
- `__init__` là **hàm khởi tạo**, chạy khi tạo đối tượng `RecentCounter()`.
- `self` là chính đối tượng đó; `self.x = ...` tạo **thuộc tính** sống suốt đời đối tượng — đây là "bộ nhớ" giữa các lần gọi method.

Ví dụ nhỏ: một bộ đếm số lần bấm.

In [1]:
class ClickCounter:
    def __init__(self):
        self.count = 0          # trạng thái ban đầu

    def click(self) -> int:
        self.count += 1         # cập nhật trạng thái
        return self.count

c1 = ClickCounter()
c2 = ClickCounter()             # đối tượng khác, trạng thái riêng
print(c1.click(), c1.click(), c1.click())
print(c2.click())

1 2 3
1


Mỗi đối tượng có `self.count` riêng: `c1` đếm tới 3, `c2` vẫn bắt đầu từ 1.

### 2.2. Cách LeetCode chạy bài thiết kế

LeetCode đưa hai danh sách: tên lệnh và tham số. Lệnh đầu là tạo đối tượng; các lệnh sau gọi method. Ta viết một **bộ chạy giả lập** để test giống LeetCode — dùng `getattr(obj, "ping")` để lấy method theo **tên chuỗi**.

In [2]:
def run_ops(cls, ops, args):
    out = []
    obj = None
    for op, a in zip(ops, args):
        if op == cls.__name__:
            obj = cls(*a)
            out.append(None)
        else:
            out.append(getattr(obj, op)(*a))
    return out

print(run_ops(ClickCounter, ["ClickCounter", "click", "click"], [[], [], []]))

[None, 1, 2]


`cls.__name__` là tên class dạng chuỗi; `cls(*a)` gọi hàm khởi tạo với các tham số trong list `a` (dấu `*` "trải" list thành tham số).

### 2.3. Hàng đợi (queue) và `collections.deque`

**Queue** là cấu trúc "vào trước ra trước" (FIFO) — như xếp hàng mua vé: người đến trước được phục vụ trước.

`deque` (double-ended queue) cho phép thêm/bớt ở **cả hai đầu** trong `O(1)`:
- `append(x)` thêm vào cuối.
- `popleft()` lấy ra từ đầu.
- `q[0]` xem phần tử đầu (không lấy ra).

In [3]:
from collections import deque

q = deque()
for x in [10, 20, 30]:
    q.append(x)
    print("append", x, "->", q)
print("đầu hàng q[0] =", q[0])
print("popleft ->", q.popleft(), "| còn", q)
print("len(q) =", len(q))

append 10 -> deque([10])
append 20 -> deque([10, 20])
append 30 -> deque([10, 20, 30])
đầu hàng q[0] = 10
popleft -> 10 | còn deque([20, 30])
len(q) = 2


### 2.4. Vì sao không dùng `list.pop(0)`?

`list.pop(0)` phải **dịch tất cả phần tử còn lại** sang trái → `O(n)`. `deque.popleft()` là `O(1)`. Đo thử:

In [4]:
import time
N = 50_000
lst = list(range(N))
t0 = time.perf_counter()
while lst:
    lst.pop(0)
t_list = time.perf_counter() - t0

dq = deque(range(N))
t0 = time.perf_counter()
while dq:
    dq.popleft()
t_deque = time.perf_counter() - t0
print(f"list.pop(0): {t_list:.4f}s | deque.popleft(): {t_deque:.4f}s")

list.pop(0): 0.2124s | deque.popleft(): 0.0026s


`50_000` — dấu gạch dưới trong số chỉ để dễ đọc, Python hiểu là `50000`.

### 2.5. `bisect` — tìm kiếm nhị phân trên list đã sắp xếp

`bisect_left(a, x)` trả vị trí **đầu tiên** mà `a[i] >= x` (nếu chèn `x` vào để giữ thứ tự, ta chèn ở đó). Thời gian `O(log n)`.

Áp dụng: các `t` tăng dần, nên số request `>= t - 3000` = `len(a) - bisect_left(a, t - 3000)`.

In [5]:
from bisect import bisect_left
a = [1, 100, 3001, 3002]
for x in [0, 1, 2, 100, 3002, 5000]:
    print(f"bisect_left(a, {x}) = {bisect_left(a, x)}")

bisect_left(a, 0) = 0
bisect_left(a, 1) = 0
bisect_left(a, 2) = 1
bisect_left(a, 100) = 1
bisect_left(a, 3002) = 3
bisect_left(a, 5000) = 4


## 3. Cách trực giác — lưu tất cả, đếm lại mỗi lần

Lưu mọi `t` vào list. Mỗi lần `ping(t)`, đếm các phần tử trong `[t - 3000, t]`.

In [6]:
class RecentCounterBrute:
    def __init__(self):
        self.times = []

    def ping(self, t: int) -> int:
        self.times.append(t)
        return sum(1 for x in self.times if t - 3000 <= x <= t)

ops = ["RecentCounter", "ping", "ping", "ping", "ping"]
args = [[], [1], [100], [3001], [3002]]
RecentCounterBrute.__name__ = "RecentCounter"   # để bộ chạy nhận diện lệnh khởi tạo
print(run_ops(RecentCounterBrute, ops, args))

[None, 1, 2, 3, 3]


`t - 3000 <= x <= t` là **so sánh chuỗi** (chained comparison) — Python hiểu là `t - 3000 <= x and x <= t`.

### 3.1. Chi phí của cách trực giác

Lần gọi thứ `k` duyệt `k` phần tử → tổng `1 + 2 + ... + n = O(n²)` với `n` lần gọi. Với `n = 10^4` là khoảng `5·10^7` phép so sánh — chậm trong Python.

In [7]:
def time_counter(cls, n):
    obj = cls()
    t0 = time.perf_counter()
    for t in range(1, n + 1):
        obj.ping(t * 7)
    return time.perf_counter() - t0

print(f"Brute n=3000: {time_counter(RecentCounterBrute, 3000):.4f}s")

Brute n=3000: 0.1835s


## 4. Nút thắt và insight

**Nút thắt:** mỗi lần đếm lại **toàn bộ lịch sử**, kể cả những request đã quá cũ.

**Quan sát:**
1. `t` tăng nghiêm ngặt → các thời điểm luôn **được thêm theo thứ tự tăng dần**.
2. Một request `x` đã "quá cũ" với `t` hiện tại (`x < t - 3000`) thì cũng **quá cũ với mọi `t` sau** (vì `t` chỉ tăng). → **Bỏ nó vĩnh viễn**.
3. Request cũ nhất luôn ở **đầu**, mới nhất ở **cuối** → đúng mô hình **hàng đợi**.

**Thuật toán:** giữ một queue các request "còn trong cửa sổ":
- `ping(t)`: `append(t)`; trong khi `q[0] < t - 3000` thì `popleft()`; trả `len(q)`.

Đây là **cửa sổ trượt** (sliding window) theo thời gian: cửa sổ `[t-3000, t]` trượt sang phải mỗi lần ping.

In [8]:
# Minh họa quan sát 2: một khi đã quá cũ thì quá cũ mãi
x = 1
for t in [3001, 3002, 5000, 9999]:
    print(f"t={t}: x={x} quá cũ? {x < t - 3000}")

t=3001: x=1 quá cũ? False
t=3002: x=1 quá cũ? True
t=5000: x=1 quá cũ? True
t=9999: x=1 quá cũ? True


## 5. Lời giải chính — deque làm cửa sổ trượt

**Mảnh 1 — `__init__`:** `self.q = deque()`.

**Mảnh 2 — thêm request:** `self.q.append(t)`.

**Mảnh 3 — đuổi request cũ:** `while self.q[0] < t - 3000: self.q.popleft()`.
Không cần kiểm tra `q` rỗng: vừa `append(t)` và `t >= t - 3000`, nên vòng `while` chắc chắn dừng trước khi lấy `t` ra.

**Mảnh 4 — trả lời:** `return len(self.q)`.

In [9]:
class RecentCounterDeque:
    def __init__(self):
        self.q = deque()

    def ping(self, t: int) -> int:
        self.q.append(t)
        while self.q[0] < t - 3000:
            self.q.popleft()
        return len(self.q)

RecentCounterDeque.__name__ = "RecentCounter"
print(run_ops(RecentCounterDeque, ops, args))

[None, 1, 2, 3, 3]


## 6. TRACE từng bước

Chuỗi `ping`: `1, 100, 3001, 3002, 6001, 6002, 9500`.

| ping(t) | t - 3000 | q sau append | bị popleft | q cuối | trả về |
|:---:|:---:|---|---|---|:---:|
| 1 | -2999 | [1] | — | [1] | 1 |
| 100 | -2900 | [1,100] | — | [1,100] | 2 |
| 3001 | 1 | [1,100,3001] | — (1 ≥ 1) | [1,100,3001] | 3 |
| 3002 | 2 | [1,100,3001,3002] | 1 | [100,3001,3002] | 3 |
| 6001 | 3001 | [...,6001] | 100 | [3001,3002,6001] | 3 |
| 6002 | 3002 | [...,6002] | 3001 | [3002,6001,6002] | 3 |
| 9500 | 6500 | [...,9500] | 3002, 6001, 6002 | [9500] | 1 |

In [10]:
def trace_recent_counter(times):
    q = deque()
    for t in times:
        q.append(t)
        removed = []
        while q[0] < t - 3000:
            removed.append(q.popleft())
        print(f"ping({t:>4}) | cửa sổ [{t - 3000:>5}, {t:>4}] | bỏ {removed!s:18} | q = {list(q)} -> {len(q)}")

trace_recent_counter([1, 100, 3001, 3002, 6001, 6002, 9500])

ping(   1) | cửa sổ [-2999,    1] | bỏ []                 | q = [1] -> 1
ping( 100) | cửa sổ [-2900,  100] | bỏ []                 | q = [1, 100] -> 2
ping(3001) | cửa sổ [    1, 3001] | bỏ []                 | q = [1, 100, 3001] -> 3
ping(3002) | cửa sổ [    2, 3002] | bỏ [1]                | q = [100, 3001, 3002] -> 3
ping(6001) | cửa sổ [ 3001, 6001] | bỏ [100]              | q = [3001, 3002, 6001] -> 3
ping(6002) | cửa sổ [ 3002, 6002] | bỏ [3001]             | q = [3002, 6001, 6002] -> 3
ping(9500) | cửa sổ [ 6500, 9500] | bỏ [3002, 6001, 6002] | q = [9500] -> 1


`{removed!s:18}` in `removed` bằng `str()` và căn rộng 18 ký tự (list không hỗ trợ căn lề trực tiếp nên cần `!s`).

Chú ý biên: ở `ping(3001)`, request `1` **vẫn được tính** vì cửa sổ là đóng `[1, 3001]` — điều kiện bỏ là `< t - 3000`, không phải `<=`.

## 7. Các lời giải khác và bảng so sánh

### 7.1. Tìm kiếm nhị phân `bisect` trên list (giữ toàn bộ lịch sử)

Vì list luôn tăng dần, đếm số phần tử `>= t - 3000` bằng `bisect_left`.

In [11]:
class RecentCounterBisect:
    def __init__(self):
        self.times = []

    def ping(self, t: int) -> int:
        self.times.append(t)
        return len(self.times) - bisect_left(self.times, t - 3000)

RecentCounterBisect.__name__ = "RecentCounter"
print(run_ops(RecentCounterBisect, ops, args))

[None, 1, 2, 3, 3]


### 7.2. List + con trỏ "đầu hàng" (không xóa thật)

Thay vì `popleft`, giữ chỉ số `self.start` của request cũ nhất còn trong cửa sổ. Tăng `start` thay cho việc xóa. Đây là cách mô phỏng queue bằng mảng mà nhiều ngôn ngữ dùng.

In [12]:
class RecentCounterPointer:
    def __init__(self):
        self.times = []
        self.start = 0

    def ping(self, t: int) -> int:
        self.times.append(t)
        while self.times[self.start] < t - 3000:
            self.start += 1
        return len(self.times) - self.start

RecentCounterPointer.__name__ = "RecentCounter"
print(run_ops(RecentCounterPointer, ops, args))

[None, 1, 2, 3, 3]


### 7.3. Bộ đệm vòng (ring buffer) kích thước cố định — khi `t` là số nguyên

Vì `t` tăng nghiêm ngặt và là số nguyên, trong cửa sổ dài 3001 đơn vị có **tối đa 3001** request. Ta có thể dùng mảng vòng cố định 3001 ô, bộ nhớ `O(1)` (hằng số 3001). Cách này dạy kỹ thuật chỉ số `% size`.

In [13]:
class RecentCounterRing:
    SIZE = 3001

    def __init__(self):
        self.buf = [0] * self.SIZE
        self.head = 0          # vị trí phần tử cũ nhất
        self.count = 0         # số phần tử đang có

    def ping(self, t: int) -> int:
        # 1) đuổi các phần tử quá cũ TRƯỚC, để chắc chắn còn chỗ trống
        while self.count > 0 and self.buf[self.head] < t - 3000:
            self.head = (self.head + 1) % self.SIZE
            self.count -= 1
        # 2) ghi t vào ô ngay sau phần tử mới nhất
        tail = (self.head + self.count) % self.SIZE
        self.buf[tail] = t
        self.count += 1
        return self.count

RecentCounterRing.__name__ = "RecentCounter"
print(run_ops(RecentCounterRing, ops, args))

[None, 1, 2, 3, 3]


Tại sao không tràn bộ đệm? Sau bước 1, mọi phần tử còn lại nằm trong `[t - 3000, t - 1]` (vì đều nhỏ hơn `t`) và là các số nguyên **khác nhau** → tối đa 3000 phần tử. Vậy khi ghi `t` luôn còn ít nhất một ô trống, và sau khi ghi có tối đa 3001 phần tử.

**Thứ tự quan trọng:** nếu ghi `t` **trước** rồi mới đuổi, khi bộ đệm đang đầy 3001 phần tử thì `tail` trùng `head` và `t` ghi đè lên phần tử cũ nhất → đếm sai. Test dày đặc ở mục 11 (`t = 1..7000`) sẽ bắt lỗi này.

### 7.4. Bảng so sánh

`n` = số lần `ping`, `W` = số request trong cửa sổ (≤ 3001).

| Cách | Time mỗi ping | Space | Ghi chú |
|---|---|---|---|
| Brute (đếm lại) | `O(n)` | `O(n)` | tổng `O(n²)` |
| **Deque** | `O(1)` khấu hao | `O(W)` | chuẩn, nên nộp |
| bisect | `O(log n)` | `O(n)` | không xóa gì |
| List + con trỏ | `O(1)` khấu hao | `O(n)` | queue bằng mảng |
| Ring buffer | `O(1)` khấu hao | `O(3001)` = `O(1)` | phụ thuộc `t` nguyên |

## 8. Code nộp LeetCode

In [14]:
from collections import deque

class RecentCounter:

    def __init__(self):
        self.q = deque()

    def ping(self, t: int) -> int:
        self.q.append(t)
        while self.q[0] < t - 3000:
            self.q.popleft()
        return len(self.q)


# Your RecentCounter object will be instantiated and called as such:
# obj = RecentCounter()
# param_1 = obj.ping(t)

assert run_ops(RecentCounter, ops, args) == [None, 1, 2, 3, 3]
print("RecentCounter PASS ví dụ đề bài:", run_ops(RecentCounter, ops, args))

RecentCounter PASS ví dụ đề bài: [None, 1, 2, 3, 3]


## 9. Vì sao thuật toán đúng?

**Invariant (sau mỗi `ping(t)`):** `q` chứa **chính xác** các request có thời điểm trong `[t - 3000, t]`, theo thứ tự tăng dần.

- **Ban đầu:** `q` rỗng — đúng.
- **Bước `ping(t)`:** trước khi gọi, `q` chứa các request trong `[t' - 3000, t']` với `t' < t`. Mọi request **không** có trong `q` đều `< t' - 3000 < t - 3000` → đúng là không thuộc cửa sổ mới (bỏ đúng).
- `append(t)`: thêm request mới, `t` thuộc cửa sổ.
- `while q[0] < t - 3000: popleft()`: vì `q` tăng dần, các phần tử quá cũ nằm liên tiếp ở **đầu** → bỏ hết chúng, dừng ở phần tử đầu tiên `>= t - 3000`; mọi phần tử sau nó cũng `>= t - 3000` và `<= t`.

Vậy `len(q)` là số request trong `[t - 3000, t]`. ∎

**Không bao giờ `IndexError`:** phần tử `t` vừa thêm luôn thỏa `t >= t - 3000`, nên `while` dừng muộn nhất tại `t`.

## 10. Phân tích Time / Space

- **Time:** mỗi request được `append` **một lần** và `popleft` **tối đa một lần** trong cả đời. Tổng công việc qua `n` lần ping là `O(n)` → **khấu hao `O(1)`** mỗi `ping`. Một lần ping riêng lẻ có thể bỏ nhiều phần tử (ví dụ `ping(9500)` bỏ 3 phần tử), nhưng trung bình vẫn `O(1)`.
- **Space:** `O(W)` với `W` = số request trong cửa sổ. Vì `t` nguyên tăng nghiêm ngặt, `W ≤ 3001` → có thể coi `O(1)` theo `n`. (Nếu không giả định `t` nguyên, `W ≤ n`.)

In [15]:
for cls in [RecentCounterBrute, RecentCounterDeque, RecentCounterBisect, RecentCounterPointer]:
    print(f"{cls.__qualname__:22} n=10000: {time_counter(cls, 10000):.4f}s")

RecentCounterBrute     n=10000: 1.6897s
RecentCounterDeque     n=10000: 0.0015s
RecentCounterBisect    n=10000: 0.0018s
RecentCounterPointer   n=10000: 0.0016s


`__qualname__` giữ tên gốc của class (ta đã đổi `__name__` thành `"RecentCounter"` để chạy bộ test giả lập).

## 11. Bộ test — mẫu đề, biên, đối chiếu

Nhóm test:
1. Ví dụ đề.
2. Một lần ping.
3. Biên đúng `t - 3000` (vẫn được tính) và `t - 3001` (bị loại).
4. Khoảng cách lớn: mọi request cũ bị loại hết.
5. Ping dày đặc `t = 1, 2, 3, ...` → kết quả bão hòa ở 3001.
6. Ngẫu nhiên có seed, đối chiếu mọi cách.

In [16]:
impls = [RecentCounterBrute, RecentCounterDeque, RecentCounterBisect, RecentCounterPointer,
         RecentCounterRing, RecentCounter]

def answers(cls, times):
    obj = cls()
    return [obj.ping(t) for t in times]

tests = [
    ([1, 100, 3001, 3002], [1, 2, 3, 3]),
    ([5], [1]),
    ([1, 3001], [1, 2]),          # 3001 - 3000 = 1 -> request 1 vẫn trong cửa sổ
    ([1, 3002], [1, 1]),          # 3002 - 3000 = 2 > 1 -> request 1 bị loại
    ([1, 2, 3, 10**9], [1, 2, 3, 1]),
    ([642, 1849, 4921, 5936, 5957], [1, 2, 1, 2, 3]),
]
for times, expected in tests:
    for cls in impls:
        got = answers(cls, times)
        assert got == expected, (cls.__qualname__, times, got, expected)

dense = list(range(1, 7001))
ref = answers(RecentCounterBrute, dense)
assert max(ref) == 3001
for cls in impls[1:]:
    assert answers(cls, dense) == ref, cls.__qualname__
print(f"PASS {len(tests)} test + test dày đặc 7000 ping x {len(impls)} cách")

PASS 6 test + test dày đặc 7000 ping x 6 cách


In [17]:
import random
rng = random.Random(933)
for _ in range(200):
    t, times = 0, []
    for _ in range(rng.randint(1, 60)):
        t += rng.randint(1, 1500)
        times.append(t)
    ref = answers(RecentCounterBrute, times)
    for cls in impls[1:]:
        assert answers(cls, times) == ref, (cls.__qualname__, times)
print("PASS 200 chuỗi ping ngẫu nhiên")

PASS 200 chuỗi ping ngẫu nhiên


## 12. Lỗi người mới hay gặp

### Lỗi 1: Dùng `<=` thay vì `<` khi loại request cũ

In [18]:
class WrongBoundary:
    def __init__(self):
        self.q = deque()
    def ping(self, t):
        self.q.append(t)
        while self.q[0] <= t - 3000:     # SAI: loại luôn request đúng biên
            self.q.popleft()
        return len(self.q)

print("WrongBoundary:", answers(WrongBoundary, [1, 100, 3001, 3002]), "| đúng: [1, 2, 3, 3]")

WrongBoundary: [1, 2, 2, 3] | đúng: [1, 2, 3, 3]


**Vì sao sai:** khoảng `[t - 3000, t]` **đóng** — request đúng tại `t - 3000` vẫn được tính. **Sửa:** `< t - 3000`.

### Lỗi 2: Khai báo queue ở mức class thay vì trong `__init__`

In [19]:
class WrongShared:
    q = deque()                          # SAI: thuộc tính class, mọi đối tượng dùng chung!
    def ping(self, t):
        self.q.append(t)
        while self.q[0] < t - 3000:
            self.q.popleft()
        return len(self.q)

a1 = WrongShared(); print(a1.ping(1), a1.ping(2))
a2 = WrongShared(); print(a2.ping(3), "<- đối tượng mới mà vẫn thấy request của a1 (đúng phải là 1)")

1 2
3 <- đối tượng mới mà vẫn thấy request của a1 (đúng phải là 1)


**Vì sao sai:** biến khai báo ngay trong thân `class` là **thuộc tính lớp**, mọi đối tượng chia sẻ. LeetCode tạo nhiều đối tượng cho nhiều test → kết quả lẫn nhau. **Sửa:** tạo trong `__init__` bằng `self.q = deque()`.

### Lỗi 3: Dùng `list.pop(0)` — đúng nhưng chậm

Như mục 2.4, `pop(0)` là `O(W)`. Với giới hạn bài này vẫn qua, nhưng là thói quen xấu; phỏng vấn sẽ hỏi.

### Lỗi 4: Quên `self.` → biến cục bộ, mất trạng thái

In [20]:
class WrongLocal:
    def __init__(self):
        q = deque()                      # biến cục bộ, biến mất khi __init__ kết thúc
    def ping(self, t):
        try:
            self.q.append(t)
        except AttributeError as e:
            return f"AttributeError: {e}"

print(WrongLocal().ping(1))

AttributeError: 'WrongLocal' object has no attribute 'q'


### Lỗi 5: Đếm bằng cách duyệt lại cả lịch sử (brute) rồi thắc mắc TLE

Đúng nhưng `O(n²)`. Hãy nhớ insight: **request quá cũ thì quá cũ mãi mãi** → xóa đi.

## 13. Mẹo nhận dạng pattern

- "Đếm/tính tổng các sự kiện trong **khoảng thời gian gần nhất**", dữ liệu đến **theo thứ tự tăng** → **queue + cửa sổ trượt**: thêm ở cuối, bỏ ở đầu.
- "Một phần tử đã hết hạn thì hết hạn mãi" là dấu hiệu dùng queue (mỗi phần tử vào/ra một lần → khấu hao `O(1)`).
- Bài **design**: trạng thái nằm trong `self.*`, khởi tạo trong `__init__`.

Bài liên quan: 346 Moving Average from Data Stream, 362 Design Hit Counter, 1429 First Unique Number, 239 Sliding Window Maximum (deque đơn điệu), 2034 Stock Price Fluctuation.

## 14. Biến thể

### 14.1. Cửa sổ có độ dài tùy chọn `W`

In [21]:
class RecentCounterW:
    def __init__(self, window: int):
        self.window = window
        self.q = deque()

    def ping(self, t: int) -> int:
        self.q.append(t)
        while self.q[0] < t - self.window:
            self.q.popleft()
        return len(self.q)

rc = RecentCounterW(10)
print([rc.ping(t) for t in [1, 5, 11, 12, 30]])   # [1, 2, 3, 3, 1]

[1, 2, 3, 3, 1]


### 14.2. Moving Average (LeetCode 346) — trung bình `size` giá trị gần nhất

Cửa sổ theo **số lượng** thay vì thời gian; giữ thêm tổng để tính nhanh.

In [22]:
class MovingAverage:
    def __init__(self, size: int):
        self.size = size
        self.q = deque()
        self.total = 0

    def next(self, val: int) -> float:
        self.q.append(val)
        self.total += val
        if len(self.q) > self.size:
            self.total -= self.q.popleft()
        return self.total / len(self.q)

ma = MovingAverage(3)
print([round(ma.next(v), 2) for v in [1, 10, 3, 5]])   # [1.0, 5.5, 4.67, 6.0]

[1.0, 5.5, 4.67, 6.0]


### 14.3. Hit Counter (LeetCode 362) — nhiều hit cùng thời điểm, truy vấn tách riêng

Khi `t` có thể **trùng nhau** và có hàm `getHits(t)` riêng, ta vẫn dùng queue; nhưng không còn giới hạn 3001 phần tử (vì trùng). Có thể nén bằng cặp `(t, số_hit)`.

In [23]:
class HitCounter:
    def __init__(self):
        self.q = deque()      # các cặp [timestamp, count]
        self.total = 0

    def hit(self, t: int) -> None:
        if self.q and self.q[-1][0] == t:
            self.q[-1][1] += 1
        else:
            self.q.append([t, 1])
        self.total += 1

    def getHits(self, t: int) -> int:
        while self.q and self.q[0][0] <= t - 300:
            self.total -= self.q.popleft()[1]
        return self.total

hc = HitCounter()
for t in [1, 2, 3]:
    hc.hit(t)
print(hc.getHits(4))
hc.hit(300)
print(hc.getHits(300), hc.getHits(301))   # 4, 3

3
4 3


Lưu ý LeetCode 362 định nghĩa cửa sổ "300 giây gần nhất" là `(t - 300, t]` nên dùng `<=` — **mỗi bài định nghĩa biên khác nhau, đọc kỹ đề!**

## 15. Bài tập tự luyện (TODO — không tự chạy)

### Bài 1 — Đoán output
```python
q = deque([1, 2, 3]); q.append(4); q.popleft(); print(q[0], len(q))
bisect_left([2, 4, 6], 5), bisect_left([2, 4, 6], 4)
```

### Bài 2 — Tự viết lại `RecentCounter` bằng deque

In [24]:
class ExerciseRecentCounter:
    def __init__(self):
        # TODO
        pass

    def ping(self, t: int) -> int:
        # TODO: append, popleft khi quá cũ, trả len
        pass

# rc = ExerciseRecentCounter(); assert [rc.ping(t) for t in [1, 100, 3001, 3002]] == [1, 2, 3, 3]

### Bài 3 — Phiên bản `bisect`

In [25]:
class ExerciseRecentCounterBisect:
    def __init__(self):
        # TODO
        pass

    def ping(self, t: int) -> int:
        # TODO: len(times) - bisect_left(times, t - 3000)
        pass

# rc = ExerciseRecentCounterBisect(); assert [rc.ping(t) for t in [1, 100, 3001, 3002]] == [1, 2, 3, 3]

### Bài 4 — Moving Average (LeetCode 346)

In [26]:
class ExerciseMovingAverage:
    def __init__(self, size: int):
        # TODO
        pass

    def next(self, val: int) -> float:
        # TODO: giữ tổng, bỏ phần tử cũ khi vượt size
        pass

# ma = ExerciseMovingAverage(3); assert [round(ma.next(v), 2) for v in [1, 10, 3, 5]] == [1.0, 5.5, 4.67, 6.0]

### Bài 5 — Đếm request trong cửa sổ cho **cả danh sách** (không dùng class)

Cho list `times` tăng nghiêm ngặt, trả list kết quả như gọi `ping` lần lượt. Dùng **hai con trỏ** (không dùng deque).

In [27]:
def exercise_recent_counts(times: list[int]) -> list[int]:
    # TODO: con trỏ left; với mỗi right, tiến left khi times[left] < times[right] - 3000
    pass

# assert exercise_recent_counts([1, 100, 3001, 3002]) == [1, 2, 3, 3]

### Bài 6 — Mini challenge: số request **lớn nhất** từng có trong một cửa sổ 3000 ms

Cho `times` tăng nghiêm ngặt, trả về giá trị lớn nhất mà `ping` từng trả.

In [28]:
def exercise_peak_load(times: list[int]) -> int:
    # TODO
    pass

# assert exercise_peak_load([1, 100, 3001, 3002, 9000]) == 3
# assert exercise_peak_load([10]) == 1

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | popleft lấy 1 ra | q = [2,3,4] | bisect_left trả vị trí đầu tiên `>= x` |
| 2 | queue các request còn hiệu lực | `self.q = deque()` | `while self.q[0] < t - 3000` |
| 3 | list tăng dần | `self.times.append(t)` | `len - bisect_left(...)` |
| 4 | cửa sổ theo số lượng | giữ `self.total` | `if len(q) > size: total -= q.popleft()` |
| 5 | deque = hai con trỏ trên mảng | `left` không lùi | `right - left + 1` |
| 6 | dùng Bài 5 | lấy `max` | `max(exercise_recent_counts(times))` |

## 17. Đáp án tham khảo

Bài 1: `q[0] = 2`, `len(q) = 3`; `bisect_left([2,4,6], 5) = 2`, `bisect_left([2,4,6], 4) = 1`.

In [29]:
q = deque([1, 2, 3]); q.append(4); q.popleft(); print(q[0], len(q))
print(bisect_left([2, 4, 6], 5), bisect_left([2, 4, 6], 4))

2 3
2 1


In [30]:
class AnswerRecentCounter:
    def __init__(self):
        self.q = deque()
    def ping(self, t):
        self.q.append(t)
        while self.q[0] < t - 3000:
            self.q.popleft()
        return len(self.q)

class AnswerRecentCounterBisect:
    def __init__(self):
        self.times = []
    def ping(self, t):
        self.times.append(t)
        return len(self.times) - bisect_left(self.times, t - 3000)

class AnswerMovingAverage:
    def __init__(self, size):
        self.size, self.q, self.total = size, deque(), 0
    def next(self, val):
        self.q.append(val); self.total += val
        if len(self.q) > self.size:
            self.total -= self.q.popleft()
        return self.total / len(self.q)

def answer_recent_counts(times):
    out, left = [], 0
    for right, t in enumerate(times):
        while times[left] < t - 3000:
            left += 1
        out.append(right - left + 1)
    return out

def answer_peak_load(times):
    return max(answer_recent_counts(times))

rc = AnswerRecentCounter(); assert [rc.ping(t) for t in [1, 100, 3001, 3002]] == [1, 2, 3, 3]
rc = AnswerRecentCounterBisect(); assert [rc.ping(t) for t in [1, 100, 3001, 3002]] == [1, 2, 3, 3]
ma = AnswerMovingAverage(3); assert [round(ma.next(v), 2) for v in [1, 10, 3, 5]] == [1.0, 5.5, 4.67, 6.0]
assert answer_recent_counts([1, 100, 3001, 3002]) == [1, 2, 3, 3]
assert answer_peak_load([1, 100, 3001, 3002, 9000]) == 3
assert answer_peak_load([10]) == 1
for times, expected in tests:
    assert answers(AnswerRecentCounter, times) == expected
    assert answer_recent_counts(times) == expected
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


## 18. Template nhớ nhanh + checklist submit

```python
from collections import deque

class RecentCounter:
    def __init__(self):
        self.q = deque()

    def ping(self, t: int) -> int:
        self.q.append(t)
        while self.q[0] < t - 3000:
            self.q.popleft()
        return len(self.q)
```

**Câu thần chú:** *"Thêm vào đuôi, đuổi khỏi đầu những ai quá cũ, độ dài hàng đợi là đáp án."*

**Checklist trước khi submit:**
- [ ] Tên class `RecentCounter`, method `ping(self, t)` đúng chữ ký.
- [ ] Trạng thái tạo trong `__init__` bằng `self.` (không đặt ở mức class).
- [ ] Dùng `deque.popleft()`, không `list.pop(0)`.
- [ ] Điều kiện loại là `< t - 3000` (khoảng đóng).
- [ ] Giải thích được khấu hao `O(1)`: mỗi request vào/ra một lần.